In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

In [25]:
pairs = [
    ("i love bread", "je mange du pain"),
]
pairs

[('i love bread', 'je mange du pain')]

In [ ]:


source_special = ["<pad>", "<unk>"]
target_special = ["<pad>", "<unk>", "<sos>", "<eos>"]

source_words = sorted(set(
    word for src, _ in pairs for word in src.split()
))
target_words = sorted(set(
    word for _, tgt in pairs for word in tgt.split()
))

In [12]:
source_words, target_words

(['bread', 'i', 'love'], ['du', 'je', 'mange', 'pain'])

In [15]:
src_vocab = {
    word: i for i, word in enumerate(source_special + source_words)
}
tgt_vocab = {
    word: i for i, word in enumerate(target_special + target_words)
}

src_idx_to_word = {i: w for w, i in src_vocab.items()}
tgt_idx_to_word = {i: w for w, i in tgt_vocab.items()}

src_vocab, tgt_vocab

({'<pad>': 0, '<unk>': 1, 'bread': 2, 'i': 3, 'love': 4},
 {'<pad>': 0,
  '<unk>': 1,
  '<sos>': 2,
  '<eos>': 3,
  'du': 4,
  'je': 5,
  'mange': 6,
  'pain': 7})

In [16]:
 src_idx_to_word, tgt_idx_to_word

({0: '<pad>', 1: '<unk>', 2: 'bread', 3: 'i', 4: 'love'},
 {0: '<pad>',
  1: '<unk>',
  2: '<sos>',
  3: '<eos>',
  4: 'du',
  5: 'je',
  6: 'mange',
  7: 'pain'})

In [18]:
input_id = [src_vocab.get(word, src_vocab.get("<unk>")) for word in "i love bread".split()]
input_id

[3, 4, 2]

In [20]:
src = torch.tensor([input_id], dtype=torch.long)

print(src)
print(src.shape)

tensor([[3, 4, 2]])
torch.Size([1, 3])


In [22]:
vocab_size = len(src_vocab)
vocab_size

5

In [36]:
embedding = nn.Embedding(vocab_size, embedding_dim=2)
embedding

Embedding(5, 2)

In [37]:
with torch.no_grad():
    embedding.weight.copy_(torch.tensor([
        [0.0, 0.0],  # <pad>
        [0.0, 0.0],  # <unk>
        [1.0, 0.0],  # i
        [0.0, 1.0],  # eat
        [1.0, 1.0],  # rice
    ]))

src_embedded = embedding(src)

print(src_embedded)
print(src_embedded.shape)

tensor([[[0., 1.],
         [1., 1.],
         [1., 0.]]], grad_fn=<EmbeddingBackward0>)
torch.Size([1, 3, 2])


In [38]:
Wx = torch.eye(2)
Wh = torch.eye(2)
bias = torch.zeros(2)

hidden = torch.zeros(1, 2)
encoder_outputs = []

for t in range(src_embedded.size(1)):
    x_t = src_embedded[:, t, :]  # [1, 2]

    hidden = torch.tanh(
        x_t @ Wx.T + hidden @ Wh.T + bias
    )

    encoder_outputs.append(hidden)

encoder_outputs = torch.stack(
    encoder_outputs, dim=1
)  # [1, 3, 2]

encoder_hidden = hidden.unsqueeze(0)  # [1, 1, 2]

print("All encoder outputs:\n", encoder_outputs)
print("Final hidden:\n", encoder_hidden)

All encoder outputs:
 tensor([[[0.0000, 0.7616],
         [0.7616, 0.9427],
         [0.9427, 0.7365]]], grad_fn=<StackBackward0>)
Final hidden:
 tensor([[[0.9427, 0.7365]]], grad_fn=<UnsqueezeBackward0>)


In [39]:
target_embedding = {
    "<sos>": torch.tensor([[0.5, 0.5]])
}

decoder_input = target_embedding["<sos>"]

print(decoder_input)
print(decoder_input.shape)

tensor([[0.5000, 0.5000]])
torch.Size([1, 2])


In [44]:
src_embedded.size(1)

3

In [48]:
Wx = torch.eye(2)
Wh = torch.eye(2)
bias = torch.zeros(2)

hidden = torch.zeros(1, 2)
encoder_outputs = []

for t in range(src_embedded.size(1)):
    print(f"Time step {t}:\nsrc_embedded:{src_embedded}\nsrc_embedded[:, t, :]:\n", src_embedded[:, t, :])
    x_t = src_embedded[:, t, :]  # [1, 2]

    hidden = torch.tanh(
        x_t @ Wx.T + hidden @ Wh.T + bias
    )

    encoder_outputs.append(hidden)
    print(encoder_outputs)

encoder_outputs = torch.stack(
    encoder_outputs, dim=1
)  # [1, 3, 2]

encoder_hidden = hidden.unsqueeze(0)  # [1, 1, 2]

print("All encoder outputs:\n", encoder_outputs)
print("Final hidden:\n", encoder_hidden)

Time step 0:
src_embedded:tensor([[[0., 1.],
         [1., 1.],
         [1., 0.]]], grad_fn=<EmbeddingBackward0>)
src_embedded[:, t, :]:
 tensor([[0., 1.]], grad_fn=<SelectBackward0>)
[tensor([[0.0000, 0.7616]], grad_fn=<TanhBackward0>)]
Time step 1:
src_embedded:tensor([[[0., 1.],
         [1., 1.],
         [1., 0.]]], grad_fn=<EmbeddingBackward0>)
src_embedded[:, t, :]:
 tensor([[1., 1.]], grad_fn=<SelectBackward0>)
[tensor([[0.0000, 0.7616]], grad_fn=<TanhBackward0>), tensor([[0.7616, 0.9427]], grad_fn=<TanhBackward0>)]
Time step 2:
src_embedded:tensor([[[0., 1.],
         [1., 1.],
         [1., 0.]]], grad_fn=<EmbeddingBackward0>)
src_embedded[:, t, :]:
 tensor([[1., 0.]], grad_fn=<SelectBackward0>)
[tensor([[0.0000, 0.7616]], grad_fn=<TanhBackward0>), tensor([[0.7616, 0.9427]], grad_fn=<TanhBackward0>), tensor([[0.9427, 0.7365]], grad_fn=<TanhBackward0>)]
All encoder outputs:
 tensor([[[0.0000, 0.7616],
         [0.7616, 0.9427],
         [0.9427, 0.7365]]], grad_fn=<StackBackw

In [49]:
target_embedding = {
    "<sos>": torch.tensor([[0.5, 0.5]])
}

decoder_input = target_embedding["<sos>"]
print(target_embedding)
print(decoder_input)
print(decoder_input.shape)

{'<sos>': tensor([[0.5000, 0.5000]])}
tensor([[0.5000, 0.5000]])
torch.Size([1, 2])


In [50]:
decoder_hidden = encoder_hidden.squeeze(0)  # [1, 2]

decoder_state = torch.tanh(
    decoder_input + decoder_hidden
)

print("Decoder state:", decoder_state)

Decoder state: tensor([[0.8942, 0.8444]], grad_fn=<TanhBackward0>)


In [57]:
encoder_projection = torch.eye(2)
decoder_projection = torch.eye(2)
score_projection = torch.tensor([1.0, 1.0])

# encoder_outputs: [1, 3, 2]
# decoder_state:   [1, 2]


In [64]:
encoder_features = encoder_outputs @ encoder_projection.T
encoder_features

tensor([[[0.0000, 0.7616],
         [0.7616, 0.9427],
         [0.9427, 0.7365]]], grad_fn=<UnsafeViewBackward0>)

In [65]:
decoder_features = decoder_state @ decoder_projection.T
# [1, 2]

combined = torch.tanh(
    encoder_features + decoder_features.unsqueeze(1)
)
# [1, 3, 2]

scores = combined @ score_projection
# [1, 3]


In [66]:
attention_weights = torch.softmax(scores, dim=1)

print("Attention weights:", attention_weights)
print("Sum:", attention_weights.sum(dim=1))

Attention weights: tensor([[0.2831, 0.3595, 0.3574]], grad_fn=<SoftmaxBackward0>)
Sum: tensor([1.], grad_fn=<SumBackward1>)


In [67]:
# attention_weights: [1, 3]
# encoder_outputs:   [1, 3, 2]

weighted_outputs = (
    attention_weights.unsqueeze(-1) * encoder_outputs
)
# [1, 3, 2]

context = weighted_outputs.sum(dim=1, keepdim=True)
# [1, 1, 2]

print("Weighted encoder outputs:\n", weighted_outputs)
print("Context vector:\n", context)
print("Context shape:", context.shape)

Weighted encoder outputs:
 tensor([[[0.0000, 0.2156],
         [0.2738, 0.3389],
         [0.3369, 0.2632]]], grad_fn=<MulBackward0>)
Context vector:
 tensor([[[0.6107, 0.8177]]], grad_fn=<SumBackward1>)
Context shape: torch.Size([1, 1, 2])


In [68]:
context_bmm = torch.bmm(
    attention_weights.unsqueeze(1),
    encoder_outputs,
)

print(context_bmm)

tensor([[[0.6107, 0.8177]]], grad_fn=<BmmBackward0>)


In [69]:
combined = torch.cat(
    [decoder_state.unsqueeze(1), context],
    dim=-1,
)

print(combined)
print(combined.shape)

tensor([[[0.8942, 0.8444, 0.6107, 0.8177]]], grad_fn=<CatBackward0>)
torch.Size([1, 1, 4])


In [71]:
output_layer = nn.Linear(
    in_features=4,
    out_features=len(tgt_vocab),
)

logits = output_layer(combined)

print("Logits shape:", logits.shape)

Logits shape: torch.Size([1, 1, 8])


In [72]:
print("Encoder outputs:", encoder_outputs.shape)
print("Decoder state:", decoder_state)
print("Scores:", scores)
print("Attention weights:", attention_weights)
print("Context:", context)
print("Combined:", combined)
print("Logits:", logits)

Encoder outputs: torch.Size([1, 3, 2])
Decoder state: tensor([[0.8942, 0.8444]], grad_fn=<TanhBackward0>)
Scores: tensor([[1.6361, 1.8751, 1.8692]], grad_fn=<UnsafeViewBackward0>)
Attention weights: tensor([[0.2831, 0.3595, 0.3574]], grad_fn=<SoftmaxBackward0>)
Context: tensor([[[0.6107, 0.8177]]], grad_fn=<SumBackward1>)
Combined: tensor([[[0.8942, 0.8444, 0.6107, 0.8177]]], grad_fn=<CatBackward0>)
Logits: tensor([[[ 0.1331,  0.2317,  0.2473, -0.4570,  0.1471,  0.0977,  0.1822,
           0.3856]]], grad_fn=<ViewBackward0>)
